In [33]:
import os, sys
sys.path.append(os.path.abspath("../../.."))

import pandas as pd
import joblib
import pandas_market_calendars as mcal

from pandas.errors import EmptyDataError
from classes.trading.actionPredictionTrading import ActionPredictionTrading
from main.classes.neural_networks.architectures.arima_model import ArimaModel

In [27]:
# --- Helpers ---
b3_cal = mcal.get_calendar('B3')

def load_full_series(csv_path: str, stock: str) -> pd.Series:
    """Carrega série completa, indexada só em pregões da B3."""
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    df.sort_index(inplace=True)
    sched = b3_cal.schedule(start_date=df.index.min(), end_date=df.index.max())
    idx   = sched.index
    return df[stock].reindex(idx).dropna()

def rolling_forecast(model_fit, series: pd.Series, step_size: int = 1) -> pd.Series:
    """Forecast roll-forward estendendo `model_fit` com valores reais."""
    preds = []
    pos = 0
    while pos < len(series):
        h = min(step_size, len(series) - pos)
        yhat = model_fit.forecast(steps=h)
        preds.extend(yhat)
        if pos + h < len(series):
            model_fit = model_fit.extend(series.iloc[pos:pos+h].values)
        pos += h
    return pd.Series(preds, index=series.index)

In [28]:
# --- Parâmetros ---
csv_path   = "../../datasets/b3_dados/processed/acoes_concat.csv"
stocks     = ["ITUB4"]
periods    = {
    "pre_pandemia":     ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia":     ("2021-09-01", "2022-09-30"),
}
arima_dir  = "../../saved_models/arima"
arima_ver  = "1.0"
shares     = 100

In [29]:
# --- Backtest sem retrain ---
results = {}

for stock in stocks:
    # carrega ARIMA treinado
    arima_path = os.path.join(arima_dir, f"{stock}_arima_v{arima_ver}.pkl")
    arima: ArimaModel = joblib.load(arima_path)

    # carrega série completa
    series = load_full_series(csv_path, stock)

    for period_name, (start, end) in periods.items():
        subset = series[start:end]
        preds  = rolling_forecast(arima.model_fit, subset, step_size=1)

        # monta DataFrame para o simulador
        df_bt = pd.DataFrame({
            'Date':      subset.index,
            'actual':    subset.values,
            'predicted': preds.values
        })

        # instancia para backtest
        ap = ActionPredictionTrading(df_bt, ticker='actual', window=1, model_path=None)
        # injeta a coluna de previsões
        ap.df['predicted'] = df_bt['predicted'].reset_index(drop=True)

        # roda as simulações
        no_sl   = ap.simulate_trading(stop_loss=False, shares_per_trade=shares)
        with_sl = ap.simulate_trading(stop_loss=True,  shares_per_trade=shares)
        bh      = ap.simulate_buy_and_hold(shares=shares)

        results[(stock, period_name)] = {
            'arima_no_stop':   no_sl,
            'arima_with_stop': with_sl,
            'arima_bh':        bh
        }

c:\Users\mathi\.conda\envs\Project_data_mining\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\Project_data_mining\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\Project_data_mining\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\Project_data_mining\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an 

In [30]:
# --- Exibe resultados ---
for (stk, per), vals in results.items():
    print(f"\n{stk} — {per.replace('_',' ').title()}")
    print("  Sem Stop Loss: ",   vals['arima_no_stop'])
    print("  Com Stop Loss: ",   vals['arima_with_stop'])
    print("  Buy & Hold:    ",   vals['arima_bh'])


ITUB4 — Pre Pandemia
  Sem Stop Loss:  {'total_return': -0.0071743507385253905, 'hit_rate': 0.5020242914979757, 'sharpe_ratio': -0.06704635473975933, 'max_drawdown': 0.012549692624307614, 'final_capital': 99282.56492614746, 'total_trades': 247, 'stop_triggered': 0}
  Com Stop Loss:  {'total_return': -0.000982422752380371, 'hit_rate': 0.5020242914979757, 'sharpe_ratio': -0.010234636300579957, 'max_drawdown': 0.007601202715877061, 'final_capital': 99901.75772476196, 'total_trades': 247, 'stop_triggered': 26}
  Buy & Hold:     {'total_return': 0.0012668800354003907, 'initial_price': 28.58014488220215, 'final_price': 29.84702491760254, 'final_capital': 100126.68800354004, 'shares_held': 100, 'days_held': 248}

ITUB4 — Durante Pandemia
  Sem Stop Loss:  {'total_return': 0.013187177658081055, 'hit_rate': 0.5266990291262136, 'sharpe_ratio': 0.056807626740575184, 'max_drawdown': 0.009053205992991283, 'final_capital': 101318.7177658081, 'total_trades': 412, 'stop_triggered': 0}
  Com Stop Loss

In [ ]:

# --- flatten dos resultados em linhas de tabela ---
flat = []
for (stock, period_name), vals in results.items():
    flat.append({
        'stock':           stock,
        'period':          period_name,
        'modelo':          'ARIMA',        
        'retorno_no_sl':   vals['arima_no_stop']['total_return'],
        'acerto_no_sl':    vals['arima_no_stop']['hit_rate'],
        'sharpe_no_sl':    vals['arima_no_stop']['sharpe_ratio'],
        'drawdown_no_sl':  vals['arima_no_stop']['max_drawdown'],
        'capital_no_sl':   vals['arima_no_stop']['final_capital'],
        'retorno_sl':      vals['arima_with_stop']['total_return'],
        'acerto_sl':       vals['arima_with_stop']['hit_rate'],
        'sharpe_sl':       vals['arima_with_stop']['sharpe_ratio'],
        'drawdown_sl':     vals['arima_with_stop']['max_drawdown'],
        'capital_sl':      vals['arima_with_stop']['final_capital'],
        'retorno_bh':      vals['arima_bh']['total_return'],
        'capital_bh':      vals['arima_bh']['final_capital'],
        'dias_bh':         vals['arima_bh']['days_held']
    })

df_results = pd.DataFrame(flat)

# --- caminho onde guardar ---
csv_path = "../../datasets/trading/arima_trading_results.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

# --- concat incremental ---
if os.path.exists(csv_path):
    try:
        df_prev = pd.read_csv(csv_path)
    except EmptyDataError:
        # arquivo existe, mas vazio: considera DataFrame vazio
        df_prev = pd.DataFrame()
    df_comb = pd.concat([df_prev, df_results], ignore_index=True)
    df_comb.drop_duplicates(subset=['stock','period','modelo'], keep='last', inplace=True)
    df_comb.to_csv(csv_path, index=False)
    print(f"Resultados ARIMA atualizados em {csv_path}")
else:
    # não existia: cria do zero
    df_results.to_csv(csv_path, index=False)
    print(f"Arquivo ARIMA criado: {csv_path}")

# carrega para visualizar
trading_results = pd.read_csv(csv_path)
trading_results



Resultados ARIMA atualizados em ../../datasets/trading/arima_trading_results.csv


,stock,period,modelo,retorno_no_sl,acerto_no_sl,sharpe_no_sl,drawdown_no_sl,capital_no_sl,retorno_sl,acerto_sl,sharpe_sl,drawdown_sl,capital_sl,retorno_bh,capital_bh,dias_bh
0,ITUB4,pre_pandemia,ARIMA,-0.007174,0.502024,-0.067046,0.012550,99282.564926,-0.000982,0.502024,-0.010235,0.007601,99901.757725,0.001267,100126.688004,248
1,ITUB4,durante_pandemia,ARIMA,0.013187,0.526699,0.056808,0.009053,101318.717766,0.037715,0.526699,0.196467,0.005190,103771.499649,-0.004545,99545.456886,413
2,ITUB4,pos_pandemia,ARIMA,0.009617,0.505576,0.079887,0.006263,100961.745262,0.013863,0.505576,0.121463,0.004910,101386.285408,-0.001864,99813.605690,270


: 